# Session 2: Database Setup + CREATE TABLE

This interactive notebook walks through the concepts, implementations, schema designs, constraints, and error diagnostic demonstrations for **Session 2: Database Setup + CREATE TABLE**.

---

## Overview of Tasks
- **Task 1:** Install & verify SQLite / MySQL database connectivity.
- **Task 2:** Create `foodie_app` database (simulating a Zomato-style backend).
- **Task 3:** Define `restaurants` table with columns (`id`, `name`, `cuisine`, `rating`, `location`).
- **Task 4:** Design and create a `users` table for a Flipkart-style app (`user_id`, `username`, `email`, `phone_number`, `created_at`).
- **Task 5:** Intentionally make an SQL syntax mistake, capture and analyze the database error message, and fix it.

### Task 1: Environment & Engine Verification
We verify that our SQLite environment is operational.

In [ ]:
import sqlite3
import os
import pandas as pd

print(f"SQLite Engine Version: {sqlite3.sqlite_version}")
os.makedirs("database", exist_ok=True)
db_path = os.path.join("database", "foodie_app.db")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
print(f"Database connection active: {db_path}")

### Task 2 & 3: Create 'restaurants' Table in 'foodie_app'
Simulating a Zomato-style backend with structured columns:
- `id`: INTEGER PRIMARY KEY AUTOINCREMENT
- `name`: VARCHAR(100) NOT NULL
- `cuisine`: VARCHAR(50) NOT NULL
- `rating`: DECIMAL(2, 1) with CHECK (rating >= 0.0 AND rating <= 5.0)
- `location`: VARCHAR(100) NOT NULL

In [ ]:
cursor.execute("DROP TABLE IF EXISTS restaurants;")

create_restaurants_sql = """
CREATE TABLE restaurants (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name VARCHAR(100) NOT NULL,
    cuisine VARCHAR(50) NOT NULL,
    rating DECIMAL(2, 1) CHECK (rating >= 0.0 AND rating <= 5.0),
    location VARCHAR(100) NOT NULL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""
cursor.execute(create_restaurants_sql)

sample_restaurants = [
    ("Haldiram's", "North Indian / Sweets", 4.3, "Connaught Place, New Delhi"),
    ("Paradise Biryani", "Hyderabadi Biryani", 4.5, "Secunderabad, Hyderabad"),
    ("Glen's Bakehouse", "Bakery / Italian", 4.6, "Indiranagar, Bengaluru"),
    ("Bademiya", "Mughlai / Kebabs", 4.1, "Colaba, Mumbai"),
    ("Peter Cat", "Continental / Indian", 4.7, "Park Street, Kolkata")
]

cursor.executemany(
    "INSERT INTO restaurants (name, cuisine, rating, location) VALUES (?, ?, ?, ?);",
    sample_restaurants
)
conn.commit()
print(f"Seeded {len(sample_restaurants)} restaurants into 'restaurants' table.")

### Task 4: Design & Create 'users' Table (Flipkart-style App)

**Architectural Decisions & Data Types:**
1. `user_id`: Primary key with auto-increment.
2. `username`: `VARCHAR(50) NOT NULL UNIQUE` - Ensures no duplicate login usernames.
3. `email`: `VARCHAR(255) NOT NULL UNIQUE` - Standard RFC 5321 length, strictly unique.
4. `phone_number`: `VARCHAR(15) NOT NULL UNIQUE` - Phone numbers must be strings to retain leading zeros and international prefixes (e.g., `+91`), avoiding integer truncation.
5. `created_at`: `TIMESTAMP DEFAULT CURRENT_TIMESTAMP` - Automatic audit creation timestamp.

In [ ]:
cursor.execute("DROP TABLE IF EXISTS users;")

create_users_sql = """
CREATE TABLE users (
    user_id INTEGER PRIMARY KEY AUTOINCREMENT,
    username VARCHAR(50) NOT NULL UNIQUE,
    email VARCHAR(255) NOT NULL UNIQUE,
    phone_number VARCHAR(15) NOT NULL UNIQUE,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""
cursor.execute(create_users_sql)

sample_users = [
    ("rahul_verma", "rahul.verma@example.com", "+919876543210"),
    ("priya_sharma", "priya.sharma@domain.in", "+919812345678"),
    ("amit_kumar", "amit.k@techmail.com", "+919701234567"),
    ("neha_patel", "neha.patel@webmail.org", "+919654321876")
]

cursor.executemany(
    "INSERT INTO users (username, email, phone_number) VALUES (?, ?, ?);",
    sample_users
)
conn.commit()
print(f"Seeded {len(sample_users)} users successfully.")

#### Testing Integrity Constraints (Duplicate Prevention)
We test attempting to insert a duplicate email to verify constraint enforcement.

In [ ]:
try:
    cursor.execute(
        "INSERT INTO users (username, email, phone_number) VALUES (?, ?, ?);",
        ("rahul_dupe", "rahul.verma@example.com", "+919999999999")
    )
except sqlite3.IntegrityError as e:
    print(f"[Integrity Error Caught as Expected]: {e}")

### Task 5: Intentional Error Generation & Resolution
We intentionally omit a comma between column specifications to observe the engine error.

In [ ]:
# Step 1: Intentionally Erroneous Statement (Missing comma)
erroneous_sql = """
CREATE TABLE test_table_error (
    id INTEGER PRIMARY KEY
    name VARCHAR(100) NOT NULL,
    cuisine VARCHAR(50) NOT NULL
);
"""

try:
    cursor.execute(erroneous_sql)
except sqlite3.OperationalError as err:
    print(f"Captured Error: {err}")
    print("Diagnostic: The SQL parser encountered unexpected token 'name' due to missing delimiter.")

# Step 2: Corrected Statement
corrected_sql = """
CREATE TABLE test_table_error (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name VARCHAR(100) NOT NULL,
    cuisine VARCHAR(50) NOT NULL
);
"""
cursor.execute("DROP TABLE IF EXISTS test_table_error;")
cursor.execute(corrected_sql)
print("Corrected SQL executed successfully!")

### Querying & Inspecting Records with Pandas

In [ ]:
df_restaurants = pd.read_sql_query("SELECT * FROM restaurants;", conn)
print("--- Restaurants (Zomato Backend) ---")
print(df_restaurants)

df_users = pd.read_sql_query("SELECT * FROM users;", conn)
print("\n--- Users (Flipkart Backend) ---")
print(df_users)

conn.close()